# TO DO:

- [ ] Add a function to split the transcript into chunks if it exceeds the context window size.
- [ ] Add a function to summarize each chunk and then combine the summaries into a final summary.
- [ ] Add a function to generate a title for the summary based on the main ideas.
- [ ] integrate the whole workflow
- [ ] check context window size for gemini pro
- [ ] check if the transcript is too long for gemini pro and split it into chunks if it is
- [ ] add counter to know how many api calls i did to not go over limit
- [ ] make deployable webapp 
- [ ] integrate into obsidian
- [ ] refine prompt
- [ ] add a a checking agent to evaluate quality of summary
- [ ] decide optimal settings quality, speed, economy.
- [ ] add possibility to add a list of links instead of just one and bulk process them
- [ ] save everything in json, maybe mangodb




https://ai.google.dev/pricing

In [2]:
import os
import json
import logging
from datetime import datetime
from urllib.parse import urlparse, parse_qs
from dotenv import load_dotenv
from youtube_transcript_api import YouTubeTranscriptApi
from googleapiclient.discovery import build
from langchain.chains import LLMChain
from langchain.prompts import ChatPromptTemplate
from langchain.schema import Document
from langchain_google_genai import ChatGoogleGenerativeAI

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# Load environment variables
load_dotenv()
YOUTUBE_API_KEY = os.getenv('YOUTUBE_API_KEY')

def get_video_id(input_value):
    """Extract the YouTube video ID from various URL formats or validate direct video ID.

    Supported formats:
    - Standard URL: https://www.youtube.com/watch?v=<video_id>
    - Shortened URL: https://youtu.be/<video_id>
    - Embed URL: https://www.youtube.com/embed/<video_id>

    Args:
        input_value (str): The YouTube URL or the video ID.

    Returns:
        str: The video ID, or None if invalid input.
    """
    
    try:
        parsed_url = urlparse(input_value)
        if parsed_url.scheme in ['http', 'https']:
            if 'youtube.com' in parsed_url.netloc:
                if parsed_url.path.startswith('/watch'):
                    return parse_qs(parsed_url.query).get('v', [None])[0]
                elif parsed_url.path.startswith('/embed/'):
                    return parsed_url.path.split('/embed/')[1].split('?')[0]
            elif 'youtu.be' in parsed_url.netloc:
                return parsed_url.path.lstrip('/')
        elif len(input_value) == 11 and input_value.isalnum():
            return input_value
    except Exception as e:
        logging.error(f"Failed to extract video ID: {e}")
    return None

def get_video_info(video_id):
    """Fetch video details using YouTube API."""
    try:
        youtube = build("youtube", "v3", developerKey=YOUTUBE_API_KEY)
        request = youtube.videos().list(part="snippet", id=video_id)
        response = request.execute()
        snippet = response["items"][0]["snippet"]
        
        #convert date format
        published_at = snippet["publishedAt"]
        formatted_date = datetime.strptime(published_at, "%Y-%m-%dT%H:%M:%SZ").strftime("%Y-%m-%d")

        return {"title": snippet["title"], "publishedAt": formatted_date}
    except Exception as e:
        logging.error(f"Error fetching video info: {e}")
        return None

def save_transcript_to_file(video_id, transcript_text, video_info, output_dir="../data/transcripts"):
    """Save transcript and video metadata to a JSON file with today's date as the save date."""
    try:
        os.makedirs(output_dir, exist_ok=True)
        file_path = os.path.join(output_dir, f"{video_id}.json")
        save_date = datetime.now().strftime("%Y-%m-%d")
        data = {
            "video_id": video_id,
            "title": video_info["title"],
            "publishedAt": video_info["publishedAt"],
            "transcript": transcript_text,
            "save_date": save_date
        }
        with open(file_path, 'w', encoding='utf-8') as file:
            json.dump(data, file, ensure_ascii=False, indent=4)
        return file_path
    except Exception as e:
        logging.error(f"Error saving transcript to file: {e}")
        return None

def extract_transcript(video_id):
    """Retrieve the YouTube transcript as a single text block."""
    try:
        transcript = YouTubeTranscriptApi.get_transcript(video_id, languages=['en', 'en-GB', 'en-US', 'en-CA', 'en-AU', 'fr', 'fr-CA'])
        return ' '.join(entry['text'] for entry in transcript)
    except Exception as e:
        logging.error(f"Error fetching transcript: {e}")
        return None

def generate_summary(transcript_text):
    """Generate a summary using LangChain."""
    try:
        llm = ChatGoogleGenerativeAI(model="gemini-1.5-pro")
        prompt = ChatPromptTemplate.from_template(
            "Give a detailed summary with the main ideas in bullet points, and ensure proper markdown formatting. Include all examples cited to illustrate each idea: {context}"
        )
        chain = prompt | llm
        result = chain.invoke({"context": [Document(page_content=transcript_text)]})
        return result.content
    except Exception as e:
        logging.error(f"Error during summarization: {e}")
        return None

def update_transcript_with_summary(file_path, summary):
    """Add the summary to the transcript JSON."""
    try:
        with open(file_path, 'r+', encoding='utf-8') as file:
            data = json.load(file)
            data.update({"summary": summary})
            file.seek(0)
            json.dump(data, file, ensure_ascii=False, indent=4)
    except Exception as e:
        logging.error(f"Error updating transcript file: {e}")

if __name__ == "__main__":
    youtube_url = input("Enter the YouTube video URL: ")
    video_id = get_video_id(youtube_url)
    if not video_id:
        logging.error("Invalid YouTube URL or video ID.")
        exit(1)

    video_info = get_video_info(video_id)
    if not video_info:
        logging.error("Failed to retrieve video info.")
        exit(1)

    transcript_text = extract_transcript(video_id)
    if not transcript_text:
        logging.error("Failed to retrieve transcript.")
        exit(1)

    file_path = save_transcript_to_file(video_id, transcript_text, video_info)
    if not file_path:
        logging.error("Failed to save transcript.")
        exit(1)

    summary = generate_summary(transcript_text)
    if summary:
        update_transcript_with_summary(file_path, summary)
        logging.info(f"Summary added to {file_path}")


2024-12-23 19:20:29,827 - INFO - file_cache is only supported with oauth2client<4.0.0
2024-12-23 19:20:45,401 - INFO - Summary added to ../data/transcripts\dFSnam97YbQ.json
